In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
#Read the dataset Q1_data.csv using read_csv()
# Load the dataset
import pandas as pd
import os
Q1_path =os.path.join(path, 'Q1_data.csv')
df_1_data =pd.read_csv(Q1_path)



In [ ]:
# Task 2: Write your code here:
#2-Inspect the first few rows using head()
print(f"Dataset shape: {df_1_data.shape}")
df_1_data.head()

In [ ]:
# Task 3: Write your code here:
#Display dataset information using info()
df_1_data.info()

In [ ]:
# Task 4: Write your code here:
#4-Show statistical description using describe()
df_1_data.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
#Plot the target distribution (delivery_time)
# Attack distribution
plt.figure(figsize=(10, 5))
plt.hist(df_1_data['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery_Time VS Preparation_Time_min Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Preparation_Time_min')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_1_data['Delivery_Time'].dropna(), bins=30, edgecolor='black', color='yellow')
plt.title('Delivery_Time VS Courier_Experience_yrs Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Courier_Experience_yrs')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_1_data['Delivery_Time'].dropna(), bins=30, edgecolor='black', color='red')
plt.title('Delivery_Time VS Distance_km	 Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Distance_km	')
plt.show()

In [ ]:
# Task 1: Write your code here: Drop the 'Order_ID' column from the data

cols = ['Order_ID', 'Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time','Traffic_Level','Weather','Time_of_Day', 'Vehicle_Type']
df_clean = df_1_data[cols].copy()
df_clean = df_clean.drop(columns=['Order_ID'])
#make sure that Orded_ID deleted
df_clean.describe()

In [ ]:
# Task 2: Write your code here: Handle missing values appropriately
# this for the data type(string or object )
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

#data type(float)
for col in ['Courier_Experience_yrs', 'Delivery_Time']:
  # Correctly assign the filled values back to the DataFrame
  df_clean[col] = df_clean[col].fillna(df_clean[col].mean())


In [ ]:
# Task 4: Write your code here:Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

categorical_features = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']

# Instantiate OneHotEncoder
onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Apply fit_transform to the categorical features
encoded_features = onehot_encoder.fit_transform(df_clean[categorical_features])

# Create a DataFrame with the encoded features
encoded_df = pd.DataFrame(encoded_features, columns=onehot_encoder.get_feature_names_out(categorical_features), index=df_clean.index)

# Drop original categorical columns and concatenate the encoded ones
df_clean = pd.concat([df_clean.drop(columns=categorical_features), encoded_df], axis=1)

print('DataFrame after One-Hot Encoding (first 5 rows):\n', df_clean.head())
print('\nShape of DataFrame after encoding:', df_clean.shape)

In [ ]:
# Task 4: Write your code here:Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder
categories= pd.DataFrame({"Q1_data": ['Traffic_Level ', 'Time_of_Day ', 'Vehicle_Type']}) # DataFrame of one column
print('data before encoding:\n', categories) #show before encoding

onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
data_onehot_encoded = onehot_encoder.fit_transform(categories) # Apply fit_transform to the copied

print('\nData after encoding:\n', data_onehot_encoded) #show after encoding

In [ ]:
# Task 5: Write your code here:Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler #import StandardScaler

num_features = ['Delivery_Time','Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs']
X_numerical = df_clean[num_features].copy()
print('Data before scaling (first 5 rows of selected features):\n', X_numerical.head()) #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
X_scaled = standard_scaler.fit_transform(X_numerical)
df_clean[num_features] = X_scaled
print('\nData after scaling (first 5 rows of scaled features):\n', df_clean[num_features].head())


In [ ]:
# Task 6: Write your code here:Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
# 1. Is the target imbalanced?
count=df_clean["Delivery_Time"].value_counts(normalize=True)
print(count)

In [ ]:
# Task 1: Write your code here:
# Split the dataset into features (X) and target (y)
X = df_clean.drop('Delivery_Time', axis=1)
y = df_clean['Delivery_Time']

# Train-test split (80% train, 20% test)
from sklearn.model_selection import train_test_split, KFold
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
 from sklearn.ensemble import RandomForestRegressor
 models={"Random Forest Regressor": RandomForestRegressor(n_estimators=200)}
 # Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold


# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
# enumerate is a built-in Python function that lets you loop over a list (or
# iterable) and get a counter automatically.
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    # .iloc does not care about the values inside the DataFrame at all — it
    # only looks at row and column positions (indices).
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print("-" * 30)

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import numpy as np
import os

#  Split the dataset into features (X) and target (y) ---
X = df_clean.drop('Delivery_Time', axis=1)
y = df_clean['Delivery_Time']
print(f"Dataset split into X (shape: {X.shape}) and y (shape: {y.shape}).")

#  Train a RandomForest model with KFold cross-validation and evaluate using MAE ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for fold, (train_index, val_index) in enumerate(kf.split(X, y)):
    print(f"\n--- Fold {fold+1}/{kf.n_splits} ---")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)
    print(f"MAE for Fold {fold+1}: {mae:.4f}")

#  Print the averaged score across all folds ---
average_mae = np.mean(mae_scores)
print(f"\nAverage MAE across all folds: {average_mae:.4f}")
print("Pipeline execution complete.")


In [ ]:
# Task 1: Plot feature importance from your trained model



feature_importances = model.feature_importances_

features_df = pd.DataFrame({'Feature': X.columns, 'Importance': feature_importances})
features_df = features_df.sort_values(by='Importance', ascending=False)

# Plotting feature importance
plt.figure(figsize=(12, 7))
plt.barh(features_df['Feature'], features_df['Importance'], color='skyblue')
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.title('Feature Importance from RandomForestRegressor')
plt.gca().invert_yaxis() # Display most important feature at the top
plt.show()


In [ ]:
# Task 2: Plot predicted delivery time histogram


# Retrain model on the full dataset (X, y) to get overall predictions
final_model = RandomForestRegressor(random_state=42)
final_model.fit(X, y)

# Get predictions for the entire dataset
all_predictions = final_model.predict(X)

# Plotting predicted delivery time histogram
plt.figure(figsize=(10, 6))
plt.hist(all_predictions, bins=30, edgecolor='black', alpha=0.7)
plt.title('Distribution of Predicted Delivery Time (Scaled)')
plt.xlabel('Predicted Delivery Time (Scaled)')
plt.ylabel('Frequency')
plt.show()

# Optionally, plot actual vs predicted for comparison
plt.figure(figsize=(10, 6))
plt.hist(y, bins=30, edgecolor='black', alpha=0.7, label='Actual (Scaled)')
plt.hist(all_predictions, bins=30, edgecolor='red', alpha=0.5, label='Predicted (Scaled)')
plt.title('Actual vs. Predicted Delivery Time Distribution (Scaled)')
plt.xlabel('Delivery Time (Scaled)')
plt.ylabel('Frequency')
plt.legend()
plt.show()


In [ ]:
# Task Bonus: Write your code here: